# Stage 1 — Zero-Shot CLIP RN50

This independent Colab notebook evaluates frozen **CLIP RN50** image and text encoders on **DTD partition 1**, **FGVC-Aircraft (variant)**, and **Oxford Flowers-102**. It uses the exact dataset-specific prompt templates from Stage 1, the complete official test splits, cosine similarity, and one result per dataset. Zero-shot CLIP uses no labeled training images and trains no parameters.

## 1. Install dependencies

In [ ]:
%pip -q install open_clip_torch scikit-learn pandas seaborn tqdm


## 2. Mount Drive and configure paths

In [ ]:
from pathlib import Path
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Flow-Matching')
else:
    PROJECT_ROOT = Path('/content/Flow-Matching')
DATA_ROOT = PROJECT_ROOT / 'data'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'zero_shot_clip'
CACHE_ROOT = PROJECT_ROOT / 'feature_cache' / 'clip_rn50'
for p in [DATA_ROOT,OUTPUT_ROOT,CACHE_ROOT]: p.mkdir(parents=True,exist_ok=True)
print('Project root:',PROJECT_ROOT)


## 3. Imports, GPU detection, and reproducibility

In [ ]:
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import open_clip

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); PIN_MEMORY=DEVICE.type=='cuda'
print('Device:',DEVICE,'| GPU:',torch.cuda.get_device_name(0) if PIN_MEMORY else 'none')
DATASETS=['dtd','aircraft','flowers102']; BATCH_SIZE=128; NUM_WORKERS=2
random.seed(0); np.random.seed(0); torch.manual_seed(0)


## 4. Official class names and required prompt templates
DTD uses `a photo of a {class} texture`, Aircraft uses `a photo of a {class} aircraft`, and Flowers-102 uses `a photo of a {class} flower`. Underscores are converted to spaces; no prompt ensembling is used.

In [ ]:
FLOWERS102_CLASSES = [
'pink primrose','hard-leaved pocket orchid','canterbury bells','sweet pea','english marigold','tiger lily','moon orchid','bird of paradise','monkshood','globe thistle',
'snapdragon','colts foot','king protea','spear thistle','yellow iris','globe-flower','purple coneflower','peruvian lily','balloon flower','giant white arum lily',
'fire lily','pincushion flower','fritillary','red ginger','grape hyacinth','corn poppy','prince of wales feathers','stemless gentian','artichoke','sweet william',
'carnation','garden phlox','love in the mist','mexican aster','alpine sea holly','ruby-lipped cattleya','cape flower','great masterwort','siam tulip','lenten rose',
'barbeton daisy','daffodil','sword lily','poinsettia','bolero deep blue','wallflower','marigold','buttercup','oxeye daisy','common dandelion',
'petunia','wild pansy','primula','sunflower','pelargonium','bishop of llandaff','gaura','geranium','orange dahlia','pink-yellow dahlia',
'cautleya spicata','japanese anemone','black-eyed susan','silverbush','californian poppy','osteospermum','spring crocus','bearded iris','windflower','tree poppy',
'gazania','azalea','water lily','rose','thorn apple','morning glory','passion flower','lotus','toad lily','anthurium',
'frangipani','clematis','hibiscus','columbine','desert-rose','tree mallow','magnolia','cyclamen','watercress','canna lily',
'hippeastrum','bee balm','ball moss','foxglove','bougainvillea','camellia','mallow','mexican petunia','bromelia','blanket flower','trumpet creeper','blackberry lily']
assert len(FLOWERS102_CLASSES)==102
PROMPT_TEMPLATES={'dtd':'a photo of a {class_name} texture','aircraft':'a photo of a {class_name} aircraft','flowers102':'a photo of a {class_name} flower'}


## 5. Load frozen CLIP RN50 and official test datasets
OpenCLIP's `openai` RN50 checkpoint and its associated validation preprocessing are used. The encoder is kept in evaluation mode with gradients disabled.

In [ ]:
model,_,preprocess=open_clip.create_model_and_transforms('RN50',pretrained='openai'); tokenizer=open_clip.get_tokenizer('RN50')
model.eval().to(DEVICE)
for p in model.parameters(): p.requires_grad_(False)
assert not any(p.requires_grad for p in model.parameters())

def build_test_dataset(name,transform):
    if name=='dtd': return datasets.DTD(DATA_ROOT,split='test',partition=1,transform=transform,download=True)
    if name=='aircraft': return datasets.FGVCAircraft(DATA_ROOT,split='test',annotation_level='variant',transform=transform,download=True)
    if name=='flowers102': return datasets.Flowers102(DATA_ROOT,split='test',transform=transform,download=True)
    raise ValueError(name)

def targets_of(ds):
    for attr in ('_labels','targets','labels'):
        if hasattr(ds,attr): return np.asarray(getattr(ds,attr),dtype=int)
    return np.asarray([ds[i][1] for i in range(len(ds))],dtype=int)

def class_names_of(name,ds):
    if name=='flowers102': return FLOWERS102_CLASSES
    if hasattr(ds,'classes'): return [str(x).replace('_',' ') for x in ds.classes]
    raise AttributeError(f'No class names exposed for {name}')

test_datasets={name:build_test_dataset(name,preprocess) for name in DATASETS}
for name,ds in test_datasets.items():
    names=class_names_of(name,ds); labels=targets_of(ds); assert len(names)==int(labels.max())+1
    print(name,len(ds),'images',len(names),'classes')


## 6. Show input samples
These are classification inputs, not generated images; Stage 1 contains no generative model.

In [ ]:
fig,axes=plt.subplots(3,5,figsize=(15,9))
for row,name in enumerate(DATASETS):
    raw=build_test_dataset(name,None); names=class_names_of(name,raw)
    for col,idx in enumerate(np.linspace(0,len(raw)-1,5,dtype=int)):
        image,label=raw[idx]; axes[row,col].imshow(image); axes[row,col].set_title(names[label][:28]); axes[row,col].axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT/'input_samples.png',dpi=180,bbox_inches='tight'); plt.show()


## 7. Cache normalized image and text embeddings

In [ ]:
@torch.inference_mode()
def encode_dataset(name,ds):
    cache=CACHE_ROOT/f'{name}__test.pt'
    if cache.exists(): return torch.load(cache,map_location='cpu',weights_only=False)
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY); zs=[]; ys=[]
    for images,labels in tqdm(loader,desc=f'CLIP RN50 / {name}'):
        zs.append(F.normalize(model.encode_image(images.to(DEVICE,non_blocking=True)).float(),dim=1).cpu()); ys.append(labels.long())
    item=dict(features=torch.cat(zs),labels=torch.cat(ys),classes=class_names_of(name,ds),dataset=name,encoder='CLIP RN50')
    assert len(item['features'])==len(ds) and torch.isfinite(item['features']).all(); torch.save(item,cache); return item

@torch.inference_mode()
def encode_text_prototypes(name,class_names):
    prompts=[PROMPT_TEMPLATES[name].format(class_name=c) for c in class_names]
    tokens=tokenizer(prompts).to(DEVICE); z=F.normalize(model.encode_text(tokens).float(),dim=1).cpu()
    torch.save(dict(features=z,classes=class_names,prompts=prompts,encoder='CLIP RN50'),CACHE_ROOT/f'{name}__text_prototypes.pt')
    return z,prompts

embedding_bank={}; text_bank={}; prompt_bank={}
for name,ds in test_datasets.items():
    embedding_bank[name]=encode_dataset(name,ds); text_bank[name],prompt_bank[name]=encode_text_prototypes(name,embedding_bank[name]['classes'])
display(pd.DataFrame([{'dataset':n,'prompt example':prompt_bank[n][0],'images':len(embedding_bank[n]['labels']),'classes':len(text_bank[n])} for n in DATASETS]))


## 8. Zero-shot cosine classification and top-1 accuracy
Because both embedding sets are unit-normalized, their dot product is cosine similarity. There is one result per dataset and no validation-based selection.

In [ ]:
rows=[]; prediction_bank={}; score_bank={}
for name in DATASETS:
    images=embedding_bank[name]['features']; labels=embedding_bank[name]['labels']; scores=images@text_bank[name].T; pred=scores.argmax(1); acc=(pred==labels).float().mean().item()
    prediction_bank[name]=pred; score_bank[name]=scores
    run_dir=OUTPUT_ROOT/'runs'/name; run_dir.mkdir(parents=True,exist_ok=True)
    np.save(run_dir/'test_predictions.npy',pred.numpy()); np.save(run_dir/'cosine_scores.npy',scores.numpy())
    metrics=dict(dataset=name,encoder='CLIP RN50',baseline='zero-shot text prototypes',test_accuracy=acc,num_test_images=len(labels),num_classes=len(text_bank[name])); (run_dir/'metrics.json').write_text(json.dumps(metrics,indent=2)); rows.append(metrics)
results_df=pd.DataFrame(rows); results_df.to_csv(OUTPUT_ROOT/'accuracy_summary.csv',index=False); display(results_df.style.format({'test_accuracy':'{:.4f}'}))
ax=sns.barplot(data=results_df,x='dataset',y='test_accuracy',color='#4c78a8'); ax.set_ylim(0,1); ax.set(title='Zero-shot CLIP RN50',ylabel='top-1 test accuracy'); ax.bar_label(ax.containers[0],fmt='%.3f'); plt.tight_layout(); plt.savefig(OUTPUT_ROOT/'accuracy.png',dpi=180,bbox_inches='tight'); plt.show()


## 9. Row-normalized confusion matrices

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(20,6))
for ax,name in zip(axes,DATASETS):
    y=embedding_bank[name]['labels'].numpy(); pred=prediction_bank[name].numpy(); sns.heatmap(confusion_matrix(y,pred,normalize='true'),cmap='mako',vmin=0,vmax=1,cbar=False,ax=ax); ax.set(title=name,xlabel='predicted class',ylabel='true class')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT/'confusion_matrices.png',dpi=180,bbox_inches='tight'); plt.show()


## 10. Joint PCA of image embeddings and text prototypes
The first 10 classes and up to 30 deterministic test examples per class are shown. PCA is fitted jointly to the displayed image embeddings and corresponding text prototypes. Class colors stay consistent within each plot; the result is qualitative.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,name in zip(axes,DATASETS):
    bank=embedding_bank[name]; y=bank['labels'].numpy(); selected=np.arange(10); idx=np.concatenate([np.flatnonzero(y==c)[:30] for c in selected]); image_z=bank['features'][idx]; text_z=text_bank[name][selected]
    xy=PCA(n_components=2).fit_transform(torch.cat([image_z,text_z]).numpy()); n=len(image_z)
    sns.scatterplot(x=xy[:n,0],y=xy[:n,1],hue=y[idx],palette='tab10',s=22,alpha=.65,legend=False,ax=ax); ax.scatter(xy[n:,0],xy[n:,1],c=selected,cmap='tab10',marker='X',s=160,edgecolors='black',label='text prototype'); ax.set(title=name,xlabel='PC1',ylabel='PC2'); ax.legend()
fig.tight_layout(); fig.savefig(OUTPUT_ROOT/'joint_image_text_pca.png',dpi=180,bbox_inches='tight'); plt.show()


## 11. Representative predictions and errors
The gallery shows confidence, ground truth, and prediction for deterministic test examples. Images are loaded from the same official test split without model preprocessing only for display.

In [ ]:
for name in DATASETS:
    raw=build_test_dataset(name,None); names=embedding_bank[name]['classes']; y=embedding_bank[name]['labels']; pred=prediction_bank[name]; confidence=score_bank[name].softmax(1).max(1).values
    correct=torch.flatnonzero(pred==y)[:4].tolist(); wrong=torch.flatnonzero(pred!=y)[:4].tolist(); chosen=correct+wrong
    fig,axes=plt.subplots(2,4,figsize=(14,7)); axes=axes.ravel()
    for ax,idx in zip(axes,chosen):
        image,_=raw[idx]; ax.imshow(image); ax.set_title(f'T: {names[y[idx]][:20]}\nP: {names[pred[idx]][:20]} ({confidence[idx]:.2f})',color='green' if pred[idx]==y[idx] else 'crimson',fontsize=9); ax.axis('off')
    for ax in axes[len(chosen):]: ax.axis('off')
    fig.suptitle(f'{name}: correct examples then errors'); fig.tight_layout(); fig.savefig(OUTPUT_ROOT/f'{name}_prediction_gallery.png',dpi=180,bbox_inches='tight'); plt.show()
print('Artifacts saved to:',OUTPUT_ROOT)
